# Task 1 — Data Preparation

**Saiket Systems | Machine Learning Internship**

Objective: load the Telco Customer Churn dataset, inspect data quality, handle missing values, convert numeric columns, and encode categorical variables for ML readiness.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Find the project root even if this notebook is opened from its own task folder.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "Dataset").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the Dataset folder. Open this notebook inside Saiket_ML_Internship.")
DATA_PATH = ROOT / "Dataset" / "Telco_Customer_Churn_Dataset.csv"
df = pd.read_csv(DATA_PATH)
print("Dataset path:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())


Dataset path: C:\Users\HP\Desktop\Saiket_ML_Internship\Dataset\Telco_Customer_Churn_Dataset.csv
Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Inspect data quality

`TotalCharges` is stored as text in the raw file. Blank strings are converted to missing values before numeric conversion.

In [2]:
print("Duplicate rows:", df.duplicated().sum())
print("\\nData types:")
display(df.dtypes.to_frame("dtype"))
df = df.drop_duplicates().copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].replace(r"^\\s*$", np.nan, regex=True), errors="coerce")
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count"))


Duplicate rows: 0
\nData types:


,dtype
customerID,str
gender,str
SeniorCitizen,int64
Partner,str
Dependents,str
tenure,int64
PhoneService,str
MultipleLines,str
InternetService,str
OnlineSecurity,str


,missing_count
TotalCharges,11


## 2. Handle missing values

For `TotalCharges`, use the median because it is a numeric feature and the median is robust to extreme values. Keep `customerID` only as an identifier; it is not a predictive feature.

In [3]:
totalcharges_median = df["TotalCharges"].median()
df["TotalCharges"] = df["TotalCharges"].fillna(totalcharges_median)
print("Filled TotalCharges missing values with median:", round(totalcharges_median, 3))
print("Missing values remaining:", int(df.isna().sum().sum()))


Filled TotalCharges missing values with median: 1397.475
Missing values remaining: 0


## 3. Encode categorical variables

The target `Churn` is encoded as 0/1. Input categorical columns use one-hot encoding, with the first category dropped to reduce redundant columns.

In [4]:
target = df["Churn"].map({"No": 0, "Yes": 1})
X_raw = df.drop(columns=["Churn", "customerID"])
X = pd.get_dummies(X_raw, drop_first=True, dtype=int)
prepared = X.copy()
prepared["Churn"] = target
print("Encoded feature matrix shape:", X.shape)
display(prepared.head())


Encoded feature matrix shape: (7043, 30)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn
0,0,1,29.85,29.85,0,1,0,0,1,0,...,0,0,0,0,0,1,0,1,0,0
1,0,34,56.95,1889.50,1,0,0,1,0,0,...,0,0,0,1,0,0,0,0,1,0
2,0,2,53.85,108.15,1,0,0,1,0,0,...,0,0,0,0,0,1,0,0,1,1
3,0,45,42.30,1840.75,1,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
4,0,2,70.70,151.65,0,0,0,1,0,0,...,0,0,0,0,0,1,0,1,0,1


## 4. Summary

The dataset is now numeric and suitable for a baseline machine-learning workflow. The original dataset remains unchanged in the `Dataset` folder; this notebook prepares data in memory.

In [5]:
print(f"Rows after duplicate removal: {len(df):,}")
print(f"Encoded input features: {X.shape[1]}")
print(f"Churn rate: {target.mean():.2%}")
print("Missing values in prepared data:", int(prepared.isna().sum().sum()))


Rows after duplicate removal: 7,043
Encoded input features: 30
Churn rate: 26.54%
Missing values in prepared data: 0
